# Request classifier — train the router's `local` backend (Kaggle, GPU T4 x2)

Trains the two-head linear model the router embeds (`llm-router/assets/request_classifier_weights.json`):
multinomial logistic regression with L2, one head for request type (7 classes, temperature-calibrated)
and one for complexity (1–5). Same objective as scikit-learn's `LogisticRegression` (lbfgs), solved with
full-batch L-BFGS in float64 on the GPUs; cross-validation folds are spread over both T4s.

Features were computed by the router's own Rust feature engine (`examples/classifier_features.rs`), so the
weights match inference exactly. Data rules:
- regularisation (C) is chosen by **grouped** 5-fold CV on train; temperature is fitted on out-of-fold
  train predictions; pruning is decided on train only;
- `val` is used **only for reporting**; the public eval sample is not in this data.

Settings: Accelerator = **GPU T4 x2**. Internet not needed. Add the `linear_features.npz` dataset as input.

## 1. Environment and input

In [ ]:
import glob, json, time, datetime, pathlib
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
from scipy.optimize import minimize_scalar
from sklearn.model_selection import GroupKFold

paths = glob.glob("/kaggle/input/**/linear_features.npz", recursive=True) or glob.glob("linear_features.npz")
assert paths, "add the dataset containing linear_features.npz via 'Add Input'"
pack = np.load(paths[0], allow_pickle=False)
LABELS = [str(x) for x in pack["labels"]]
NUM_BUCKETS, NUM_DENSE = int(pack["num_buckets"]), int(pack["num_dense"])
DEVICES = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
print("devices:", DEVICES, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), (
    "No GPU found. In the right panel open Session options -> Accelerator -> GPU T4 x2, then Run All again. "
    "On CPU this takes hours.")
OUTPUT = pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle/working").exists() else pathlib.Path(".")

## 2. Configuration

In [ ]:
C_GRID_TYPE = C_GRID_CX = [4.0, 8.0, 16.0, 32.0, 64.0, 128.0, 256.0]
FOLDS = 5
MAX_ITER = 500           # L-BFGS iterations per fit
PRUNE_THRESHOLDS = [0.0, 0.01, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5]
MIN_AGREEMENT = 0.995    # pruned model must agree with the unpruned one on >= 99.5% of TRAIN examples
DECIMALS = 4
ECE_BINS = 10

## 3. Load features

In [ ]:
import scipy.sparse as sp

def matrix(split):
    n = len(pack[f"{split}_id"])
    return sp.csr_matrix((pack[f"{split}_values"].astype(np.float64), pack[f"{split}_indices"], pack[f"{split}_indptr"]),
                         shape=(n, NUM_BUCKETS + NUM_DENSE))

X_full, Xv_full = matrix("train"), matrix("val")
y_type, y_cx = pack["train_type"].astype(np.int64), pack["train_complexity"].astype(np.int64)
yv_type, yv_cx = pack["val_type"].astype(np.int64), pack["val_complexity"].astype(np.int64)
groups = pack["train_group"]
val_ids = [str(x) for x in pack["val_id"]]

# Fit only columns that occur in train (unseen columns get weight 0 either way).
used = np.unique(np.concatenate([X_full.indices, np.arange(NUM_BUCKETS, NUM_BUCKETS + NUM_DENSE)]))
X, Xv = X_full[:, used].tocsr(), Xv_full[:, used].tocsr()
print(f"train {X.shape[0]} x {X.shape[1]} active cols, val {Xv.shape[0]}; nnz {X.nnz}")

## 4. Solver and metrics

In [ ]:
def to_torch(Xs, device):
    Xs = Xs.tocsr()
    return torch.sparse_csr_tensor(torch.from_numpy(Xs.indptr).long(), torch.from_numpy(Xs.indices).long(),
                                   torch.from_numpy(Xs.data), size=Xs.shape, dtype=torch.float64, device=device)

class Fitted:
    def __init__(self, W, b, classes):
        self.coef_, self.intercept_, self.classes = W, b, classes
    def decision_function(self, Xs):
        return np.asarray(Xs @ self.coef_.T) + self.intercept_

def fit_lr(Xs, y, C, classes, device):
    """argmin  C * sum_i CE(x_i W^T + b, y_i) + 0.5 * ||W||^2   (bias unpenalised, as in scikit-learn)."""
    Xt = to_torch(Xs, device)
    yt = torch.from_numpy(y).to(device)
    W = torch.zeros(Xs.shape[1], classes, dtype=torch.float64, device=device, requires_grad=True)
    b = torch.zeros(classes, dtype=torch.float64, device=device, requires_grad=True)
    opt = torch.optim.LBFGS([W, b], lr=1, max_iter=MAX_ITER, history_size=20, tolerance_grad=1e-7,
                            tolerance_change=1e-10, line_search_fn="strong_wolfe")
    def closure():
        opt.zero_grad()
        logits = torch.sparse.mm(Xt, W) + b
        loss = C * torch.nn.functional.cross_entropy(logits, yt, reduction="sum") + 0.5 * (W * W).sum()
        loss.backward()
        return loss
    opt.step(closure)
    return Fitted(W.detach().cpu().numpy().T.copy(), b.detach().cpu().numpy().copy(), classes)

def softmax(logits, T=1.0):
    z = logits / T; z = z - z.max(1, keepdims=True); e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def ece(probs, y):
    conf, correct, total = probs.max(1), probs.argmax(1) == y, 0.0
    for lo in np.linspace(0, 1, ECE_BINS, endpoint=False):
        m = (conf > lo) & (conf <= lo + 1 / ECE_BINS)
        if m.any():
            total += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(total)

def fit_temperature(logits, y):
    nll = lambda T: -np.log(softmax(logits, T)[np.arange(len(y)), y] + 1e-12).mean()
    return float(minimize_scalar(nll, bounds=(0.05, 20.0), method="bounded").x)

accuracy = lambda y, logits: float((logits.argmax(1) == y).mean())
neg_mae = lambda y, logits: -float(np.abs(logits.argmax(1) - y).mean())

## 5. Grouped cross-validation over C (folds spread over both GPUs) + temperature

In [ ]:
FOLD_SPLITS = list(GroupKFold(n_splits=FOLDS).split(X, y_type, groups))

def oof_logits(y, C, classes):
    """Out-of-fold logits; folds run concurrently, one per GPU in turn."""
    def run(k):
        tr, te = FOLD_SPLITS[k]
        clf = fit_lr(X[tr], y[tr], C, classes, DEVICES[k % len(DEVICES)])
        return te, clf.decision_function(X[te])
    out = np.zeros((X.shape[0], classes))
    with ThreadPoolExecutor(max_workers=len(DEVICES)) as pool:
        for te, logits in pool.map(run, range(FOLDS)):
            out[te] = logits
    return out

def pick_C(y, classes, score, name, grid):
    scores, logits = {}, {}
    for C in grid:
        t = time.time()
        logits[C] = oof_logits(y, C, classes)
        scores[C] = score(y, logits[C])
        print(f"{name}: C={C:<5} oof={scores[C]:.4f}  ({time.time() - t:.0f}s)")
    best = max(scores, key=scores.get)
    return best, scores, logits[best]

C_type, type_scores, type_oof = pick_C(y_type, len(LABELS), accuracy, "type", C_GRID_TYPE)
T = fit_temperature(type_oof, y_type)
print(f"type: C={C_type}, temperature={T:.3f}, oof ECE {ece(softmax(type_oof), y_type):.4f} -> {ece(softmax(type_oof, T), y_type):.4f}")
C_cx, cx_scores, _ = pick_C(y_cx, 5, neg_mae, "complexity", C_GRID_CX)
print(f"complexity: C={C_cx}")

## 6. Final fit, train-only pruning, validation report

In [ ]:
type_clf = fit_lr(X, y_type, C_type, len(LABELS), DEVICES[0])
cx_clf = fit_lr(X, y_cx, C_cx, 5, DEVICES[-1])

def expand(clf, classes):
    coef = np.zeros((classes, NUM_BUCKETS + NUM_DENSE)); coef[:, used] = clf.coef_
    return coef, clf.intercept_

def pruned(coef, threshold):
    out = coef.copy()
    hashed = out[:, :NUM_BUCKETS]
    hashed[:, np.abs(hashed).max(0) < threshold] = 0.0
    return np.round(out, DECIMALS)

def choose_threshold(coef, bias, y_labels):
    """Largest pruning threshold whose predictions on TRAIN still agree with the unpruned model."""
    full = np.asarray(X_full @ coef.T) + bias
    best = 0.0
    for t in PRUNE_THRESHOLDS:
        p = np.asarray(X_full @ pruned(coef, t).T) + bias
        if (p.argmax(1) == full.argmax(1)).mean() >= MIN_AGREEMENT:
            best = t
    return best

type_coef, type_bias = expand(type_clf, len(LABELS))
cx_coef, cx_bias = expand(cx_clf, 5)
t_type, t_cx = choose_threshold(type_coef, type_bias, y_type), choose_threshold(cx_coef, cx_bias, y_cx)
type_coef, cx_coef = pruned(type_coef, t_type), pruned(cx_coef, t_cx)
print("prune thresholds:", t_type, t_cx)

# Validation: reporting only.
vt = np.asarray(Xv_full @ type_coef.T) + type_bias
vc = np.asarray(Xv_full @ cx_coef.T) + cx_bias
val_metrics = {
    "type_accuracy": accuracy(yv_type, vt),
    "type_ece": ece(softmax(vt, T), yv_type),
    "complexity_exact": float((vc.argmax(1) == yv_cx).mean()),
    "complexity_mae": float(np.abs(vc.argmax(1) - yv_cx).mean()),
}
print("validation:", json.dumps(val_metrics, indent=1))
print("per-label recall:", {LABELS[k]: round(float((vt.argmax(1)[yv_type == k] == k).mean()), 3) for k in range(len(LABELS))})

## 7. Write the weights file

In [ ]:
def head_json(coef, bias):
    hashed = coef[:, :NUM_BUCKETS]
    keep = np.nonzero(np.abs(hashed).max(0) > 0)[0]
    return {"bias": [round(float(v), DECIMALS) for v in bias],
            "dense": [[float(w) for w in coef[:, NUM_BUCKETS + f]] for f in range(NUM_DENSE)],
            "hashed": {str(int(b)): [float(w) for w in hashed[:, b]] for b in keep}}

weights = {
    "schema": "nasiko-request-classifier-linear-v1",
    "num_buckets": NUM_BUCKETS, "num_dense_features": NUM_DENSE, "request_types": LABELS,
    "temperature": round(T, 5),
    "type_head": head_json(type_coef, type_bias),
    "complexity_head": head_json(cx_coef, cx_bias),
    "provenance": {
        "trained_at_utc": datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "training_data_sha256": str(pack["train_sha256"]),
        "train_examples": int(X.shape[0]),
        "type_C": C_type, "complexity_C": C_cx,
        "oof_type_accuracy_by_C": {str(k): round(v, 4) for k, v in type_scores.items()},
        "oof_complexity_neg_mae_by_C": {str(k): round(v, 4) for k, v in cx_scores.items()},
        "prune_thresholds": {"type": t_type, "complexity": t_cx},
        "validation": {k: round(v, 4) for k, v in val_metrics.items()},
        "trainer": "llm-router/classifier/scripts/train_linear_gpu.ipynb (L2 multinomial LR, full-batch L-BFGS float64)",
        "feature_engine": "llm-router/src/routing/classifier/linear.rs (dumped by examples/classifier_features.rs)",
    },
}
out = OUTPUT / "request_classifier_weights.json"
out.write_text(json.dumps(weights, separators=(",", ":")) + "\n", encoding="utf-8")
(OUTPUT / "linear_results.json").write_text(json.dumps(weights["provenance"], indent=1), encoding="utf-8")
print(f"wrote {out} ({out.stat().st_size / 1e6:.2f} MB; {len(weights['type_head']['hashed'])} + "
      f"{len(weights['complexity_head']['hashed'])} buckets)")
print("Download request_classifier_weights.json and linear_results.json from the Output panel.")